<a href="https://colab.research.google.com/github/saiakhi1l/Behavior-Aware-Rider-Risk-Scoring-System-using-Trajectory-Data/blob/main/customer_support_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installing Dependencies

In [ ]:
!pip install pandas
!pip install numpy
!pip install chromadb
!pip install sentence-transformers
!pip install rank-bm25
!pip install groq
!pip install langchain-text-splitters
!pip install scikit-learn
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentel

In [ ]:
!pip install langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetr

In [ ]:
!pip install guardrails-
# for this project we are not using this module we are doinfg manually

ERROR: Invalid requirement: 'guardrails-': Expected end or semicolon (after name and no valid version specifier)
    guardrails-
              ^


KeyboardInterrupt: 

In [ ]:
import os
import zipfile
import time

import pandas as pd
import numpy as np

import chromadb

from groq import Groq

from tqdm import tqdm

from rank_bm25 import BM25Okapi

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from sklearn.model_selection import train_test_split

#zip extraction

In [ ]:
zip_path = "/content/customer_it_support.zip"

extract_path = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted successfully.")

ZIP extracted successfully.


In [ ]:
all_files=[]
for root, folder, files in os.walk(extract_path):
  for file in files:
    if file.endswith(".csv"):
      file_path=os.path.join(root,file)
      all_files.append(file_path)
print(all_files)

['/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', '/content/data/dataset-tickets-german_normalized.csv', '/content/data/dataset-tickets-multi-lang3-4k.csv', '/content/data/dataset-tickets-multi-lang-4-20k.csv', '/content/data/dataset-tickets-german_normalized_50_5_2.csv']


In [ ]:
from langchain_community.document_loaders import CSVLoader

/tmp/ipykernel_1406/3306274919.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


#Data Ingestion

In [ ]:
documents=[]
metadata=[]
for file in all_files:
  loader=CSVLoader(file)
  docs=loader.load()
  for doc in docs:
    documents.append(doc.page_content)
    metadata.append(doc.metadata)

# print(f"Documents{documents}")
print(f"metadata{metadata}")

metadata[{'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 0}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 1}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 2}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 3}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 4}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 5}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 6}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 7}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 8}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 9}, {'source': '/content/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'row': 10}, {'source': '/content/d

In [ ]:
for f in all_files:
    if "aa_dataset" in f:
        df = pd.read_csv(f)
        break

In [ ]:
print(df.columns.tolist())

['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [ ]:
print(df["priority"].unique())
print(df["language"].unique())
print(df["queue"].unique())

['high' 'medium' 'low']
['de' 'en']
['Technical Support' 'Returns and Exchanges' 'Billing and Payments'
 'Sales and Pre-Sales' 'Service Outages and Maintenance' 'Product Support'
 'IT Support' 'Customer Service' 'Human Resources' 'General Inquiry']


#Data Cleaning

In [ ]:
#NULL PERCENTAGE

for col in df.columns:
    null_percent=(df[col].isnull().sum()/df.shape[0])*100
    print(f"{col} - {null_percent:.2f}%")

subject - 13.43%
body - 0.00%
answer - 0.02%
type - 0.00%
queue - 0.00%
priority - 0.00%
language - 0.00%
version - 0.00%
tag_1 - 0.00%
tag_2 - 0.05%
tag_3 - 0.48%
tag_4 - 10.70%
tag_5 - 49.12%
tag_6 - 79.45%
tag_7 - 92.86%
tag_8 - 98.02%


In [ ]:

df["tag_5"]=df["tag_5"].fillna("Unknown")

df["tag_6"]=df["tag_6"].fillna("Unknown")

In [ ]:
df["tag_7"]=df["tag_7"].fillna("Unknown")

In [ ]:
df["tag_7"].isnull().sum()

np.int64(0)

In [ ]:
df=df.drop(columns=["tag_8"])

In [ ]:
print(df.columns)

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language',
       'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6',
       'tag_7'],
      dtype='object')


#Document Construction + Metadata Extraction

In [ ]:
documents=[]
metadata=[]

for _,row in df.iterrows(): # here iter rows returns ids and rows here we dont want ids so we used"_" so it skips ids

    doc=f"""
Subject: {row['subject']}
Body: {row['body']}
Answer: {row['answer']}
"""

    documents.append(doc)

    metadata.append({
        "priority":row["priority"],
        "language":row["language"],
        "type":row["type"],
        "queue":row["queue"]
    })

In [ ]:
documents

['\nSubject: Wesentlicher Sicherheitsvorfall\nBody: Sehr geehrtes Support-Team,\\n\\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\\n\\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.\nAnswer: Vielen Dank für die Meldung des kritischen Sicherheitsvorfalls und die Bereitstellung der Übersicht über die betroffenen Geräte sowie der ergriffenen ersten Maßnahmen. Wir erkennen di

#Chunking

In [ ]:
#chunking

splitter=RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

all_chunks=[]
all_meta=[]

for doc, meta in zip(documents,metadata):
  chunks=list(splitter.split_text(doc))
  for chunk in chunks:
    all_chunks.append(chunk)
    all_meta.append(meta)
all_chunks

['Subject: Wesentlicher Sicherheitsvorfall\nBody: Sehr geehrtes Support-Team,\\n\\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\\n\\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.\nAnswer: Vielen Dank für die Meldung des kritischen Sicherheitsvorfalls und die Bereitstellung der Übersicht über die betroffenen Geräte sowie der ergriffenen ersten Maßnahmen. Wir erkennen die 

In [ ]:
print(len(all_chunks))

28590


#Embeddings

In [ ]:
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model.device

In [ ]:

embeddings=model.encode(all_chunks,
                        batch_size=256,
                        show_progress_bar=True,
                        normalize_embeddings=True).tolist()

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

In [ ]:
import numpy as np

#BM25 Indexing

In [ ]:
# BM25 INDEXING ------used for only key word searching

tokenized_chunks=[]
for chunk in all_chunks:
  tokenized_chunks.append(chunk.lower().split())
bm25=BM25Okapi(tokenized_chunks)

#ChromaDB Indexing

In [ ]:

# CHROMA DB INDEXING


import chromadb
client = chromadb.Client()
# Delete old collection if it exists
try:
    client.delete_collection(
        name="it_support"
    )
except:
    pass
# Create fresh collection
collection = client.create_collection(
    name="it_support"
)
# Create unique IDs
ids = [
    f"id{i}"
    for i in range(len(all_chunks))
]

# Batch insert because Chroma
# cannot handle very large inserts
batch_size = 4000

for i in range(
    0,
    len(all_chunks),
    batch_size
):

    collection.add(
        documents=all_chunks[
            i:i+batch_size
        ],

        embeddings=embeddings[
            i:i+batch_size
        ],

        metadatas=all_meta[
            i:i+batch_size
        ],

        ids=ids[
            i:i+batch_size
        ]
    )

    print(
        f"Inserted "
        f"{min(i+batch_size,len(all_chunks))}"
        f" / {len(all_chunks)} chunks"
    )

print(
    "Chroma indexing completed."
)



Inserted 4000 / 28590 chunks
Inserted 8000 / 28590 chunks
Inserted 12000 / 28590 chunks
Inserted 16000 / 28590 chunks
Inserted 20000 / 28590 chunks
Inserted 24000 / 28590 chunks
Inserted 28000 / 28590 chunks
Inserted 28590 / 28590 chunks
Chroma indexing completed.
Total chunks in Chroma: 28590


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.66MiB/s]


{'ids': [['id531', 'id356', 'id4982']], 'embeddings': None, 'documents': [['Subject: Login Problems with Windows 10\nBody: Dear Customer Support,\\n\\nI am facing difficulties when trying to access my Windows 10 Pro device. Each login attempt results in the system either becoming unresponsive or showing an error message that blocks access. I have rebooted several times and verified that my keyboard functions correctly, but the issue remains. This problem is significantly affecting my productivity as I cannot reach my files or applications. Could you please help me resolve this login issue as soon as possible? Here are the troubleshooting steps I have tried.\nAnswer: Thank you for reaching out regarding your login difficulties with your Windows 10 Pro device. To assist you better, could you specify the exact error message you encounter during login? Additionally, are you attempting to log in with a local account or a Microsoft account? If possible, try booting your computer into Safe Mo

In [ ]:
# RERANKER
reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
# simple CACHE
cache={}

In [ ]:

# CONFIGURATION


import os
import numpy as np
from groq import Groq

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

cache = {}

INSTRUCTION_PHRASES = [
    "show",
    "list",
    "find",
    "give me",
    "summarize",
    "high priority",
    "medium priority",
    "low priority",
    "english",
    "german",
    "technical support tickets",
    "it support tickets",
    "tickets related to",
    "related to"
]

# QUERY CLEANIN

def extract_topic_query(raw_query):
    q = raw_query.lower()
    for phrase in INSTRUCTION_PHRASES:
        q = q.replace(phrase, " ")
    q = " ".join(q.split())
    return q if q else raw_query


# ==========================================
# RECIPROCAL RANK FUSION (RRF)
# ==========================================

def rrf_fuse(
    bm25_chunks,
    vector_chunks,
    bm25_meta,
    vector_meta,
    k=60
):

    scores = {}
    chunk_map = {}
    meta_map = {}

    for rank, (chunk, meta) in enumerate(
        zip(bm25_chunks, bm25_meta)
    ):

        scores[chunk] = (
            scores.get(chunk, 0)
            + 1 / (k + rank + 1)
        )

        chunk_map[chunk] = chunk
        meta_map[chunk] = meta

    for rank, (chunk, meta) in enumerate(
        zip(vector_chunks, vector_meta)
    ):

        scores[chunk] = (
            scores.get(chunk, 0)
            + 1 / (k + rank + 1)
        )

        chunk_map[chunk] = chunk
        meta_map[chunk] = meta

    fused = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    fused_chunks = [
        chunk_map[c]
        for c, _ in fused
    ]

    fused_meta = [
        meta_map[c]
        for c, _ in fused
    ]

    return fused_chunks, fused_meta



# CONTEXT BUILDER

def build_context(
    chunks,
    metadatas
):

    context_blocks = []

    for chunk, meta in zip(
        chunks,
        metadatas
    ):

        header = (
            f"[Priority: {meta.get('priority','unknown')} | "
            f"Queue: {meta.get('queue','unknown')} | "
            f"Language: {meta.get('language','unknown')}]"
        )

        context_blocks.append(
            f"{header}\n{chunk}"
        )

    return "\n\n".join(context_blocks)



# MAIN QUERY LOOP

while True:

    raw_query = input(
        "Enter Query: "
    )

    query = raw_query.lower()

    if query == "exit":
        break

    if query in cache:
        print(cache[query])
        continue



    # METADATA FILTER EXTRACTION


    filters = {}

    if "high priority" in query:
        filters["priority"] = "high"

    if "medium priority" in query:
        filters["priority"] = "medium"

    if "low priority" in query:
        filters["priority"] = "low"

    if "english" in query:
        filters["language"] = "en"

    if "german" in query:
        filters["language"] = "de"

    if "technical support" in query:
        filters["queue"] = "Technical Support"

    if "it support" in query:
        filters["queue"] = "IT Support"



    # QUERY PREPROCESSING


    retrieval_query = extract_topic_query(
        query
    )



    # BM25 RETRIEVAL


    bm25_scores = bm25.get_scores(
        retrieval_query.split()
    )

    bm25_indices = np.argsort(
        bm25_scores
    )[-30:][::-1]

    bm25_chunks = [
        all_chunks[i]
        for i in bm25_indices
    ]

    bm25_meta = [
        all_meta[i]
        for i in bm25_indices
    ]



    # BM25 METADATA FILTERING


    filtered_chunks = []
    filtered_meta = []

    for chunk, meta in zip(
        bm25_chunks,
        bm25_meta
    ):

        keep = True

        for key, value in filters.items():

            if meta.get(key) != value:
                keep = False
                break

        if keep:
            filtered_chunks.append(chunk)
            filtered_meta.append(meta)

    bm25_chunks = filtered_chunks
    bm25_meta = filtered_meta



    # VECTOR RETRIEVAL (CHROMADB)


    query_embedding = model.encode(
        [retrieval_query],
        normalize_embeddings=True
    ).tolist()

    where_clause = (
        {
            "$and": [
                {k: v}
                for k, v in filters.items()
            ]
        }
        if filters
        else None
    )

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=30,
        where=where_clause
    )

    vector_chunks = results["documents"][0]
    vector_meta = results["metadatas"][0]


    # HYBRID RETRIEVAL USING RRF


    hybrid_chunks, hybrid_meta = rrf_fuse(
        bm25_chunks,
        vector_chunks,
        bm25_meta,
        vector_meta
    )


    # DEDUPLICATION

    unique = []
    seen = set()

    for chunk, meta in zip(
        hybrid_chunks,
        hybrid_meta
    ):

        key = (
            chunk,
            str(meta)
        )

        if key not in seen:

            unique.append(
                (chunk, meta)
            )

            seen.add(key)

    hybrid_chunks = [
        x[0]
        for x in unique
    ]

    hybrid_meta = [
        x[1]
        for x in unique
    ]



    # CROSS ENCODER RERANKING


    pairs = [
        [retrieval_query, chunk]
        for chunk in hybrid_chunks
    ]

    scores = reranker.predict(
        pairs
    )

    reranked = sorted(
        zip(
            hybrid_chunks,
            hybrid_meta,
            scores
        ),
        key=lambda x: x[2],
        reverse=True
    )



    # TOP-K CONTEXT SELECTION


    top_chunks = [
        x[0]
        for x in reranked[:5]
    ]

    top_meta = [
        x[1]
        for x in reranked[:5]
    ]

    context = build_context(
        top_chunks,
        top_meta
    )



    # LLM PROMPT GENERATION


    prompt = f"""
You are a senior IT support analyst.

Use ONLY the provided context.

For each ticket provide:

- Ticket Title
- Root Cause
- Resolution Steps
- Priority
- Ticket Type

Only write 'Not specified'
if the information is unavailable.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""



    # LLM INFERENCE


    response = (
        groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2
        )
    )

    answer = (
        response
        .choices[0]
        .message.content
    )



    # CACHE STORAGE


    cache[query] = answer



    # FINAL OUTPUT


    print("\nANSWER:\n")
    print(answer)

Retrieval query used: login failures and .

BM25 TOP RESULTS (before metadata filtering)

BM25 1
Subject: Frequent Login Issues Noted During Peak Times
Body: Users are experiencing intermittent login failures during peak hours. This might be due to server overload or authentication issues. Despite increasing server capacity and restarting services, the problem continues. Please investigate and resolve the issue as soon as possible to ensure a smooth user experience.
Answer: Received an email regarding intermittent login failures during peak times. Please investigate the issue urgently to de
--------------------------------------------------
BM25 2
Subject: Support Inquiry for Occasional Login Issues
Body: Dear Customer Support, I am encountering login difficulties during peak hours. The problem might be due to server overload or database contention. Despite restarting the server and optimizing queries, the issue continues, leading to login failures and operational disruptions. Could yo